In [ ]:
%%writefile fmm3d.h
#pragma once           // il file viene incluso una sola volta
#include <cstdio>    
#include <vector>
#include <cmath>      // fminf, sqrtf
#include <queue>       // std::priority_queue
#include <functional>  // std::greater
#include <utility>     // std::pair
#include <chrono>      // misura dei tempi
#include <fstream>     // scrittura su file
const float INF = 1e30f;
typedef std::pair<float, int> Entry; // coppia (u, indice) per la coda di priorita'

__host__ __device__ float solve_quadratic3(float a, float b, float c,float F, float h)
{
    float Fh = F * h;
    if (a > b) { float t = a; a = b; b = t; }      // ordina: a <= b <= c
    if (b > c) { float t = b; b = c; c = t; }
    if (a > b) { float t = a; a = b; b = t; }

    float u = a + Fh;                              // 1 vicino (caso 1)
    if (u <= b) return u;
    u = 0.5f * (a + b + sqrtf(2.0f * Fh * Fh - (a - b) * (a - b)));   // 2 vicini (caso 2 = Alg. 2)
    if (u <= c) return u;
    float d = 3.0f * Fh * Fh - ((a - b) * (a - b) + (b - c) * (b - c) + (c - a) * (c - a));
    return (a + b + c + sqrtf(d)) / 3.0f;          // 3 vicini (caso 3)                                  
}

float solve_node(const std::vector<float>& u, const std::vector<float>& F,
                 int k, int i, int j, int N, float h)
{
    int idx = (k * N + i) * N + j;  // indice globale del nodo (k, i, j)
    // a: min dei vicini lungo x (stessa riga, colonna j-1 e j+1)
    float left  = (j > 0)     ? u[idx - 1] : INF;
    float right = (j < N - 1) ? u[idx + 1] : INF;
    // b: min dei vicini lungo y (stessa colonna, riga i-1 e i+1)
    float down  = (i > 0)     ? u[idx - N] : INF;
    float up    = (i < N - 1) ? u[idx + N] : INF;
    // c: min dei vicini lungo z (stessa colonna e riga, piano k-1 e k+1)
    float back  = (k > 0)     ? u[(idx - N * N)] : INF;
    float front = (k < N - 1) ? u[(idx + N *N)] : INF;

    // Per arrivare devi scorrere come un indice globale: u[i][j][k] = u[(k * N + i) * N + j]
    float a = fminf(left, right);
    float b = fminf(down, up);
    float c = fminf(back, front);

    float q = solve_quadratic3(a, b, c, F[idx], h);
    return fminf(u[idx], q);                                  // return min(u_ijk, u)
}
void fmm(std::vector<float>& u, const std::vector<float>& F, int N, float h, const std::vector<int>& srcs, const std::vector<unsigned char>& mask)
{
    enum { LONTANO = 0, ATTIVO = 1, NOTO = 2 };      // stato di ogni nodo
    std::vector<char> state(N * N * N, LONTANO);    // array di char NxNxN, inizializzato a LONTANO

    std::priority_queue<Entry, std::vector<Entry>, std::greater<Entry>> heap;
    for (int s : srcs) {                             // tutte le sorgenti sono i primi Trial
        heap.push(Entry(u[s], s));
        state[s] = ATTIVO;
    }

    const int dk[6] = { 0, 0, 0, 0, -1, 1 };              // spostamenti verso i 6 vicini:
    const int di[6] = {  0, 0, -1, 1, 0, 0 };              // spostamenti verso i 6 vicini:
    const int dj[6] = { -1, 1,  0, 0, 0, 0 };              // sinistra, destra, sotto, sopra

    while (!heap.empty()) {                          // while A != vuoto
        Entry trial = heap.top();                    // Trial <- radice del min-heap
        heap.pop();                                  // Rimuovi Trial da A
        int x = trial.second;                        // indice del nodo Trial
        if (state[x] == NOTO || trial.first > u[x])  // coppia "scaduta": si ignora
            continue;
        state[x] = NOTO;                             // Aggiungi Trial a Set

        int k = x / (N*N), i = (x / N) % N, j = x % N;                    // strato, riga, colonna del nodo Trial
        for (int d = 0; d < 6; ++d) {                // foreach x_nb adiacente a Trial
            int ni = i + di[d], nj = j + dj[d], nk = k + dk[d];
            if (ni < 0 || ni >= N || nj < 0 || nj >= N || nk < 0 || nk >= N) continue;   // fuori dal dominio
            int nb = (nk * N + ni) * N + nj;   // indice del nodo vicino
            if (state[nb] == NOTO || !mask[nb]) continue;         // già noto, oppure ostacolo

            float q = solve_node(u, F, nk, ni, nj, N, h);
            if (q < u[nb]) {                         // u(x_nb) <- soluzione di g(x_nb)
                u[nb] = q;
                state[nb] = ATTIVO;                  // Aggiungi x_nb ad A
                heap.push(Entry(q, nb));
            }
        }
    }
}
template <class Esatta>
double relative_error(const std::vector<float>& u, int N, float h, Esatta esatta)
{
    double num = 0.0, den = 0.0;
    for (int k = 0; k < N; ++k)
        for (int i = 0; i < N; ++i)
            for (int j = 0; j < N; ++j) {
            double x = -1.0 + j * h, y = -1.0 + i * h, z = -1.0 + k * h;   // coordinate del nodo (k, i, j)
            double U = esatta(x, y, z);                         // soluzione esatta
            double d = u[(k * N + i) * N + j] - U;
            num += d * d;                               // ||u - U||^2
            den += U * U;                               // ||U||^2
            }
    return std::sqrt(num / den);    // Norma euclidea relativa: ||u - U|| / ||U||
}


In [ ]:
%%writefile ambiente.h
#pragma once           // il file viene incluso una sola volta
#include <vector>
#include <cmath>

// Forme con cui si descrive l'ambiente (coordinate nel cubo [-1, 1]^3)
struct Punto   { double x, y, z; };
struct Scatola { double x0, x1, y0, y1, z0, z1; };       // parallelepipedo: x0 <= x <= x1, y0 <= y <= y1, z0 <= z <= z1
struct Sfera   { double cx, cy, cz, r; };                 // centro (cx, cy, cz) e raggio r

// =====================================================================
// L'AMBIENTE: per cambiarlo si modifica solo questa parte
// =====================================================================
const double MARGINE = 0.06;                             // distanza minima dagli ostacoli (raggio del drone + sicurezza)

const Punto A = { -0.85, -0.85, -0.85 };                 // partenza del drone: angolo in basso
const Punto B = {  0.85,  0.85,  0.85 };                 // arrivo (sorgente del FIM): angolo opposto, in alto

const std::vector<Scatola> SCATOLE = {                   // {x0, x1, y0, y1, z0, z1}
    { -0.45, -0.35, -1.0, 1.0, -1.0, 1.0 },              // muro 1 (x = -0.4)
    {  0.15,  0.25, -1.0, 1.0, -1.0, 1.0 },              // muro 2 (x = 0.2)
    {  0.45,  0.65,  0.2, 1.0,  0.2, 1.0 },              // blocco davanti a B: si entra solo da sotto o di lato
    { -1.0,  -0.45, -0.3, -0.2, -1.0, 0.4 },             // paratia tra A e il muro 1
};
const std::vector<Sfera> SFERE = {                       // {cx, cy, cz, r}
    { -0.1,  0.3, -0.4, 0.25 },                          // sfera tra i due muri, davanti alla finestra 1
    { -0.1, -0.4,  0.4, 0.22 },                          // sfera tra i due muri, davanti alla finestra 2
};
const std::vector<Scatola> FORI = {                      // zone in cui l'ostacolo viene tolto
    { -0.45, -0.35,  0.5,  0.8, -0.8, -0.5 },            // finestra nel muro 1: y alta, z bassa
    {  0.15,  0.25, -0.8, -0.5,  0.5,  0.8 },            // finestra nel muro 2: y bassa, z alta
};
// =====================================================================

inline bool dentro(const Scatola& s, double x, double y, double z)
{
    return s.x0 <= x && x <= s.x1 && s.y0 <= y && y <= s.y1 && s.z0 <= z && z <= s.z1;
}

inline bool dentro(const Sfera& s, double x, double y, double z)
{
    return (x - s.cx) * (x - s.cx) + (y - s.cy) * (y - s.cy) + (z - s.cz) * (z - s.cz) <= s.r * s.r;
}

// Vero se il punto (x, y, z) e' dentro u)
inline bool ostacolo(double x, double y, double z)
{
    for (const auto& f : FORI)    if (dentro(f, x, y, z)) return false;
    for (const auto& s : SCATOLE) if (dentro(s, x, y, z)) return true;
    for (const auto& s : SFERE)   if (dentro(s, x, y, z)) return true;
    return false;
}

// Maschera della griglia N^3: 1 = nodo l ostacolo (resta INF)
inline std::vector<unsigned char> costruisci_maschera(int N)
{
    const double h = 2.0 / (N - 1);
    std::vector<unsigned char> mask((size_t)N * N * N);
    for (int k = 0; k < N; ++k)                              // 1) gli ostacoli veri
        for (int i = 0; i < N; ++i)
            for (int j = 0; j < N; ++j)
                mask[((size_t)k * N + i) * N + j] = ostacolo(-1.0 + j * h, -1.0 + i * h, -1.0 + k * h) ? 0 : 1;

    // 2) Margine di sicurezza: diventa ostacolo ogni nodo a distanza <= MARGINE da un nodo ostacolo
    const int r = (int)std::ceil(MARGINE / h);               // raggio del margine, in nodi
    std::vector<unsigned char> allargata = mask;
    for (int k = 0; k < N; ++k)
        for (int i = 0; i < N; ++i)
            for (int j = 0; j < N; ++j) {
                if (mask[((size_t)k * N + i) * N + j]) continue;                       // si parte solo dai nodi ostacolo...
                bool bordo = false;                        // ...che toccano unnodo libero
                if (j > 0     && mask[((size_t)k * N + i) * N + j - 1]) bordo = true;
                if (j < N - 1 && mask[((size_t)k * N + i) * N + j + 1]) bordo = true;
                if (i > 0     && mask[((size_t)k * N + i - 1) * N + j]) bordo = true;
                if (i < N - 1 && mask[((size_t)k * N + i + 1) * N + j]) bordo = true;
                if (k > 0     && mask[((size_t)(k - 1) * N + i) * N + j]) bordo = true;
                if (k < N - 1 && mask[((size_t)(k + 1) * N + i) * N + j]) bordo = true;
                if (!bordo) continue;
                for (int dk = -r; dk <= r; ++dk)                  // tutti i nodi entro la sfera di raggio MARGINE
                    for (int di = -r; di <= r; ++di)
                        for (int dj = -r; dj <= r; ++dj) {
                            const int kk = k + dk, ii = i + di, jj = j + dj;
                            if (kk < 0 || kk >= N || ii < 0 || ii >= N || jj < 0 || jj >= N) continue;
                            if ((dk * dk + di * di + dj * dj) * h * h <= MARGINE * MARGINE)
                                allargata[((size_t)kk * N + ii) * N + jj] = 0;
                        }
            }
    return allargata;
}


inline bool ambiente_valido()
{
    return !ostacolo(A.x, A.y, A.z) && !ostacolo(B.x, B.y, B.z);
}

In [ ]:
%%writefile fim3d.cu
#include <cstdio>          
#include <cstdlib>         
#include <cuda_runtime.h>
#include <algorithm>       // std::min
#include <chrono>          // tempi sulla CPU
#include <fstream>         // salvataggio su file
#include "fmm3d.h"           // Fast Marching
#include "ambiente.h"       // Ostacoli

// Controllo degli errori CUDA: se la chiamata fallisce, stampa file, riga e motivo ed esce.
#define CUDA_CHECK(call)                                                     \
    do {                                                                     \
        cudaError_t err = (call);                                            \
        if (err != cudaSuccess) {                                            \
            printf("Errore CUDA %s:%d: %s\n", __FILE__, __LINE__,            \
                   cudaGetErrorString(err));                                 \
            exit(1);                                                         \
        }                                                                    \
    } while (0)

#define BLOCK 8  // dimensione del blocco 8x8x8 = 512 thread
__device__ float solve_in_block(const float s[BLOCK + 2][BLOCK + 2][BLOCK + 2], int z, int y, int x, float F, float h)
{
    float a = fminf(s[z][y][x - 1], s[z][y][x + 1]);               // a <- min dei vicini lungo x
    float b = fminf(s[z][y - 1][x], s[z][y + 1][x]);               // b <- min dei vicini lungo y
    float c = fminf(s[z - 1][y][x], s[z + 1][y][x]);               // c <- min dei vicini lungo z
    return fminf(s[z][y][x], solve_quadratic3(a, b, c, F, h));      // return min(u_ijk, u)
}

__device__ void process_block(int block, int niter, float* u, const float* F, const unsigned char* mask,
                             int N, int nbx, float h, float eps, unsigned char* Cb_out)
{
    __shared__ float s_u[BLOCK + 2][BLOCK + 2][BLOCK + 2];                // block + halo, nella shared memory
    __shared__ unsigned char s_c[BLOCK * BLOCK * BLOCK];                  // i flag Cn dei 512 nodi

    int bi = (block / nbx) % nbx, bj = block % nbx, bk = block / (nbx * nbx);            // riga, colonna e strato del blocco
    int tx = threadIdx.x, ty = threadIdx.y, tz = threadIdx.z;
    int tid = (tz * BLOCK + ty) * BLOCK + tx;                                       // numero del thread nel blocco: 0..511
    int i = bi * BLOCK + ty, j = bj * BLOCK + tx, k = bk * BLOCK + tz;              // nodo di questo thread
    int idx = (k * N + i) * N + j;                                                  // posizione del nodo nella griglia
    bool inside = (i < N && j < N && k < N);                          // l'ultimo blocco puo' sporgere
    bool fixed  = !inside || !mask[idx];             // fuori dominio non aggiornare nodi
    float f = inside ? F[idx] : 1.0f;                  // F_ijk del nodo

    // Global -> Shared: 1000 celle (block + halo), 512 thread
    for (int m = tid; m < (BLOCK + 2) * (BLOCK + 2) * (BLOCK + 2); m += BLOCK * BLOCK * BLOCK) {
        int sz = m / ((BLOCK + 2) * (BLOCK + 2));
        int sy = (m / (BLOCK + 2)) % (BLOCK + 2);
        int sx = m % (BLOCK + 2);                             // cella (sz, sy, sx) della shared memory
        int gi = bi * BLOCK + sy - 1;                         // nodo della griglia corrispondente:
        int gj = bj * BLOCK + sx - 1;                         
        int gk = bk * BLOCK + sz - 1;                         // -1 perche' lo strato/riga/colonna 0 e' l'halo
        bool in_domain = (gi >= 0 && gi < N && gj >= 0 && gj < N && gk >= 0 && gk < N);
        s_u[sz][sy][sx] = in_domain ? u[(gk * N + gi) * N + gj] : INF;      // fuori dal dominio: INF
    }
    __syncthreads();                                         // la block e' completa prima di usarla

    // niter iterazioni locali in shared memory
    bool conv = true;                                        // Cn del nodo
    for (int it = 0; it < niter; ++it) {
        float p = s_u[tz+1][ty + 1][tx + 1];                 // valore prima dell'aggiornamento
        float q = fixed ? p                                  // fuori dominio: non cambia
                        : solve_in_block(s_u, tz+1, ty + 1, tx + 1, f, h);
        __syncthreads();                                     // (A) tutti hanno letto i vicini...
        s_u[tz+1][ty + 1][tx + 1] = q;
        __syncthreads();                                     // (B) ...e tutti hanno scritto
        conv = fabsf(p - q) < eps;                           // Cn: il nodo non e' cambiato
    }

    // Shared -> Global: ogni thread riscrive il PROPRIO nodo (non l'halo)
    if (inside) u[idx] = s_u[tz+1][ty + 1][tx + 1];

    // Cb(b) <- reduction(Cn(b)): AND dei 512 flag
    s_c[tid] = conv;
    __syncthreads();
    for (int stride = BLOCK * BLOCK * BLOCK / 2; stride > 0; stride /= 2) {     // 256, ..., 4, 2, 1
        if (tid < stride) s_c[tid] = s_c[tid] && s_c[tid + stride];
        __syncthreads();
    }
    if (tid == 0) Cb_out[block] = s_c[0];                     // vero solo se TUTTI i nodi sono convergenti
}

__global__ void update_active_blocks(const int* list, float* u, const float* F, const unsigned char* mask,
                                    int N, int nbx, float h, int niter,
                                    float eps, unsigned char* Cb1)
{
    process_block(list[blockIdx.x], niter, u, F, mask, N, nbx, h, eps, Cb1);
}

__global__ void build_active_list(const unsigned char* Cb, unsigned char* active,
                                  int* list, int* count, int n_blocks) 
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;           // numero del blocco di nodi
    if (t >= n_blocks) return;                               // thread in eccesso
    bool act = !Cb[t];                                       // attivo se NON convergente
    active[t] = act;
    if (act) list[atomicAdd(count, 1)] = t;                  // aggiungi t alla lista A
}

__global__ void build_candidate_list(const int* list, int count, int nbx, const unsigned char* active,
                                     const unsigned char* Cb1, unsigned char* Cb,
                                     int* cand, int* cand_list, int* cand_count)
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;           // blockDim = 256
    if (t >= count) return;                                  // thread in eccesso
    int b = list[t];                                         // un blocco della lista A
    Cb[b] = Cb1[b];                                          // non convergente (Caso 1): resta in A
    if (!Cb1[b]) return;                                     // proseguono solo i blocchi appena convergenti

    int bk = b / (nbx * nbx), bi = (b / nbx) % nbx, bj = b % nbx;                          // strato, riga e colonna del blocco
    int  nb[6] = { b - 1,  b + 1,       b - nbx, b + nbx,  b - nbx * nbx, b + nbx * nbx};  // sinistra, destra, sotto, sopra e davanti, dietro
    bool ok[6] = { bj > 0, bj < nbx - 1, bi > 0,  bi < nbx - 1, bk > 0, bk < nbx - 1 };
    for (int d = 0; d < 6; ++d) {
        if (!ok[d]) continue;                                // fuori dalla griglia di blocchi
        if (active[nb[d]] && !Cb1[nb[d]]) continue;          // Caso 1: lo aggiorna già lo Step 1
        if (atomicExch(&cand[nb[d]], 1) == 0)                // primo a segnarlo: niente doppioni
            cand_list[atomicAdd(cand_count, 1)] = nb[d];
    }
}

__global__ void update_candidate_blocks(const int* cand_list, const int* cand_count, int* cand,
                                        float* u, const float* F, const unsigned char* mask, int N, int nbx, float h, float eps, unsigned char* Cb)
{
    if (blockIdx.x >= *cand_count) return;                   // blocco CUDA in eccesso
    int b = cand_list[blockIdx.x];
    process_block(b, 1, u, F, mask, N, nbx, h, eps, Cb);      // test del blocco: 1 iterazione
    if (threadIdx.x == 0 && threadIdx.y == 0 && threadIdx.z == 0) cand[b] = 0;   // pronto per il giro successivo
}

int block_fim(std::vector<float>& u, const std::vector<float>& F, int N, float h,
              const std::vector<int>& srcs, const std::vector<unsigned char>& mask, float eps, int niter, float* ms)
{
    const int nbx = (N + BLOCK - 1) / BLOCK, n_blocks = nbx * nbx * nbx; // nbx = numero di blocchi per riga/colonna, n_blocks = numero totale di blocchi
    const size_t bytes = (size_t)N * N * N * sizeof(float); // dimensione in byte della griglia

    std::vector<unsigned char> Cb0(n_blocks, 1);    // flag Cb iniziali: 1 = convergente, 0 = attivo
    // I blocchi che contengono una sorgente partono attivi Cb0 = 0
    for (int s : srcs) {
        const int sk = s / (N * N), si = (s / N) % N, sj = s % N;   // strato, riga e colonna del nodo sorgente
        Cb0[((sk / BLOCK) * nbx + si / BLOCK) * nbx + sj / BLOCK] = 0;   // il blocco di questa sorgente parte attivo
    }

    // Memoria sulla GPU e dati iniziali
    float *d_u, *d_F; unsigned char *d_Cb, *d_Cb1, *d_active; int *d_list, *d_count;
    int *d_cand, *d_cand_list, *d_cand_count; unsigned char* d_mask;
    CUDA_CHECK(cudaMalloc(&d_u, bytes));
    CUDA_CHECK(cudaMalloc(&d_F, bytes));
    CUDA_CHECK(cudaMalloc(&d_mask, (size_t)N * N * N));
    CUDA_CHECK(cudaMalloc(&d_Cb, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_Cb1, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_active, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_count, sizeof(int)));

    // Memoria per i blocchi candidati (vicini dei blocchi appena convergenti)
    CUDA_CHECK(cudaMalloc(&d_cand, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_cand_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_cand_count, sizeof(int)));

    CUDA_CHECK(cudaMemcpy(d_u, u.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_F, F.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_mask, mask.data(), (size_t)N * N * N, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_Cb, Cb0.data(), n_blocks, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemset(d_Cb1, 0, n_blocks));
    CUDA_CHECK(cudaMemset(d_cand, 0, n_blocks * sizeof(int)));


    const int threads = 256, list_blocks = (n_blocks + threads - 1) / threads;   // numero di blocchi per costruire la lista A
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    CUDA_CHECK(cudaEventRecord(start));                      // "segna l'ora" prima del ciclo
    int iter = 0;
    while (true) {
        int count = 0;
        CUDA_CHECK(cudaMemset(d_count, 0, sizeof(int)));
        build_active_list<<<list_blocks, threads>>>(d_Cb, d_active, d_list, d_count, n_blocks);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaMemcpy(&count, d_count, sizeof(int), cudaMemcpyDeviceToHost));
        if (count == 0) break;  // se count == 0, lista A vuota, termina ciclo
        ++iter;
        // Step 1: foreach b in A in parallelo
        update_active_blocks<<<count, dim3(BLOCK, BLOCK, BLOCK)>>>(d_list, d_u, d_F, d_mask, N, nbx, h, niter, eps, d_Cb1);    // si lanciano count blocchi CUDA, ognuno con 512 thread
        CUDA_CHECK(cudaGetLastError());
        // Step 2: solo i vicini dei blocchi appena convergenti
        CUDA_CHECK(cudaMemset(d_cand_count, 0, sizeof(int)));
        build_candidate_list<<<(count + threads - 1) / threads, threads>>>(d_list, count, nbx, d_active, d_Cb1, d_Cb,
                                                                          d_cand, d_cand_list, d_cand_count);
        CUDA_CHECK(cudaGetLastError());
        update_candidate_blocks<<<std::min(6 * count, n_blocks), dim3(BLOCK, BLOCK, BLOCK)>>>(d_cand_list, d_cand_count, d_cand,
                                                                                    d_u, d_F, d_mask, N, nbx, h, eps, d_Cb);
        CUDA_CHECK(cudaGetLastError());
    }
    CUDA_CHECK(cudaEventRecord(stop));                       // "segna l'ora" dopo il ciclo
    CUDA_CHECK(cudaEventSynchronize(stop));                  // aspetta che la GPU ci arrivi
    CUDA_CHECK(cudaEventElapsedTime(ms, start, stop));       // tempo trascorso [ms]
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));

    // Risultato sull'host e pulizia
    CUDA_CHECK(cudaMemcpy(u.data(), d_u, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_u)); CUDA_CHECK(cudaFree(d_F));CUDA_CHECK(cudaFree(d_Cb));
    CUDA_CHECK(cudaFree(d_Cb1)); CUDA_CHECK(cudaFree(d_mask));
    CUDA_CHECK(cudaFree(d_active)); CUDA_CHECK(cudaFree(d_list)); CUDA_CHECK(cudaFree(d_count));
    CUDA_CHECK(cudaFree(d_cand)); CUDA_CHECK(cudaFree(d_cand_list)); CUDA_CHECK(cudaFree(d_cand_count));
    return iter;
}

int nodo(int N, double x, double y, double z){
    const double h = 2.0 / (N - 1);
    const int j = (int)lround((x + 1.0) / h), i = (int)lround((y + 1.0) / h), k = (int)lround((z + 1.0) / h);
    return (k * N + i) * N + j;
}

int main()
{
    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("GPU: %s | multiprocessori: %d | max thread per blocco: %d | shared memory per blocco: %zu KB\n",
           prop.name, prop.multiProcessorCount, prop.maxThreadsPerBlock, prop.sharedMemPerBlock / 1024);
    
    CUDA_CHECK(cudaFree(0));                                 // inizializza la GPU prima delle misure
    const float eps = 1e-6f;                                 // tolleranza di convergenza
    const int niter = 3 * BLOCK;                              // 24 iterazioni locali
    const int ripetizioni = 3;                               // ogni tempo = il minimo di 3 misure
    
     if (!ambiente_valido()) {                                // A o B dentro un ostacolo: il problema non ha senso
        printf("Errore: A o B sono dentro un ostacolo, correggi ambiente.h\n");
        return 1;
    }
    printf("Ambiente di ambiente.h: sorgente in B = (%.2f, %.2f, %.2f), partenza A = (%.2f, %.2f, %.2f)\n",
           B.x, B.y, B.z, A.x, A.y, A.z);
    printf("    N   giri   max|FIM - FMM|   u(A) FIM   u(A) FMM   NaN   ostacoli non INF   t FMM [ms]   t FIM [ms]   speedup\n");

    for (int Nk : { 33, 65, 129, 193, 257 }) {
        const float hk = 2.0f / (Nk - 1);                    // griglia (§2.4)
        const int n = Nk * Nk * Nk;                          // numero di nodi
        const std::vector<unsigned char> mask = costruisci_maschera(Nk);    // 1 = libero, 0 = ostacolo
        const std::vector<int> srcs = { nodo(Nk, B.x, B.y, B.z) };          // la sorgente e' l'arrivo B
        const int iA = nodo(Nk, A.x, A.y, A.z);              // nodo della partenza A

        std::vector<float> Fk(n, 1.0f);                      // F = 1: u = lunghezza del percorso piu' corto fino a B
        std::vector<float> u0(n, INF);                       // condizione iniziale
        for (int s : srcs) u0[s] = 0.0f;

        std::vector<float> uk_fmm, uk_fim;
        double t_fmm = 1e30, t_fim = 1e30;
        int it = 0;
        for (int r = 0; r < ripetizioni; ++r) {
            uk_fmm = u0;                                     // ogni misura riparte da capo
            auto t0 = std::chrono::high_resolution_clock::now();
            fmm(uk_fmm, Fk, Nk, hk, srcs, mask);             // CPU: Algoritmo 3
            auto t1 = std::chrono::high_resolution_clock::now();
            t_fmm = std::min(t_fmm, std::chrono::duration<double, std::milli>(t1 - t0).count());

            uk_fim = u0;
            float ms = 0.0f;
            it = block_fim(uk_fim, Fk, Nk, hk, srcs, mask, eps, niter, &ms);   // GPU: Algoritmo 7
            t_fim = std::min(t_fim, (double)ms);
        }

        // Controlli: FIM = FMM, nessun NaN, gli ostacoli restano INF
        float diff = 0.0f;
        int n_nan = 0, ost_non_inf = 0;
        for (int q = 0; q < n; ++q) {
            diff = fmaxf(diff, fabsf(uk_fim[q] - uk_fmm[q]));
            if (std::isnan(uk_fim[q])) ++n_nan;
            if (!mask[q] && uk_fim[q] < INF) ++ost_non_inf;
        }
        int non_raggiunti = 0;                               // nodi liberi che l'onda non ha raggiunto
        for (int q = 0; q < n; ++q) if (mask[q] && uk_fim[q] >= INF) ++non_raggiunti;
        if (uk_fim[iA] >= INF)                               // l'onda partita da B non e' mai arrivata ad A
            printf("%5d   %4d   %14.2e   A NON RAGGIUNGIBILE da B (nodi liberi non raggiunti: %d)\n", Nk, it, diff, non_raggiunti);
        else
            printf("%5d   %4d   %14.2e   %8.4f   %8.4f   %3d   %16d   %10.1f   %10.2f   %7.1f\n", Nk, it, diff,
                   uk_fim[iA], uk_fmm[iA], n_nan, ost_non_inf, t_fmm, t_fim, t_fmm / t_fim);

        if (Nk == 65) {                                      // per i grafici (Python): soluzione, maschera, A e B
            std::ofstream out("fim_65_ostacoli.bin", std::ios::binary);
            out.write(reinterpret_cast<const char*>(uk_fim.data()), uk_fim.size() * sizeof(float));
            std::ofstream om("mask_65.bin", std::ios::binary);
            om.write(reinterpret_cast<const char*>(mask.data()), mask.size());
            std::ofstream ab("AB.txt");
            ab << A.x << " " << A.y << " " << A.z << "\n" << B.x << " " << B.y << " " << B.z << "\n";
        }
    }
    
    return 0;
}


In [ ]:
!nvcc -O3 -arch=sm_75 fim3d.cu -o fim3d && ./fim3d

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator
from skimage.measure import marching_cubes

# Dati scritti dal programma C++ (N = 65): soluzione del FIM, maschera degli ostacoli, punti A e B
N = 65
h = 2.0 / (N - 1)
u = np.fromfile("fim_65_ostacoli.bin", dtype=np.float32).reshape(N, N, N).astype(np.float64)   # u[k, i, j] = u(z, y, x)
mask = np.fromfile("mask_65.bin", dtype=np.uint8).reshape(N, N, N)                             # 1 = libero, 0 = ostacolo
A, B = np.loadtxt("AB.txt")                                                                     # coordinate (x, y, z)
x = -1.0 + h * np.arange(N)

libero = mask == 1
def nodo(p):                                                        # nodo (k, i, j) piu' vicino al punto p = (x, y, z)
    return tuple(np.round((np.asarray(p)[::-1] + 1.0) / h).astype(int))
def coord(n):                                                       # coordinate (x, y, z) del nodo n = (k, i, j)
    return -1.0 + h * np.array(n[::-1], dtype=float)

if u[nodo(A)] >= 1e29:                                              # u(A) = INF: l'onda partita da B non e' mai arrivata ad A
    raise SystemExit("A non e' raggiungibile da B: nessun percorso (controlla ostacoli e finestre in ambiente.h)")

# 1) Discesa sui nodi: da A si passa sempre al vicino libero (tra i 26 attorno) con u piu' piccolo, fino a B
mosse = [(dk, di, dj) for dk in (-1, 0, 1) for di in (-1, 0, 1) for dj in (-1, 0, 1) if (dk, di, dj) != (0, 0, 0)]
n, nB = nodo(A), nodo(B)
nodi = [n]
while n != nB:
    vicini = [(n[0] + a, n[1] + b, n[2] + c) for a, b, c in mosse]
    vicini = [m for m in vicini if all(0 <= t < N for t in m) and libero[m]]
    n = min(vicini, key=lambda m: u[m])                             # il vicino piu' vicino a B
    nodi.append(n)

# 2) Accorciamento: da ogni punto si salta al nodo piu' avanti che si "vede" senza attraversare ostacoli
peso = RegularGridInterpolator((x, x, x), libero.astype(float))     # = 1 solo se gli 8 nodi attorno sono liberi
def si_vede(p, q):                                                  # il segmento p-q non tocca ostacoli?
    t = np.linspace(0.0, 1.0, int(np.linalg.norm(q - p) / (h / 4)) + 2)[:, None]
    punti = p + t * (q - p)
    return np.all(peso(punti[:, ::-1]) > 0.999)
P = [coord(m) for m in nodi]
percorso = [P[0]]
a = 0
while a < len(P) - 1:
    b = len(P) - 1
    while b > a + 1 and not si_vede(P[a], P[b]):                    # il nodo piu' lontano visibile da P[a]
        b -= 1
    percorso.append(P[b])
    a = b
percorso = np.array(percorso)
u_interp = RegularGridInterpolator((x, x, x), np.where(libero, u, 0.0))
def u_in(p):                                                        # u in un punto qualsiasi: media dei soli nodi liberi attorno
    q = np.asarray(p)[::-1]
    return u_interp(q)[0] / peso(q)[0]

lunghezza = np.sum(np.linalg.norm(np.diff(percorso, axis=0), axis=1))
iA = nodo(A)
print(f"u(A) = {u[iA]:.4f}   lunghezza del percorso = {lunghezza:.4f}   linea retta |A - B| = {np.linalg.norm(A - B):.4f}")
print(f"{len(nodi)} nodi nella discesa, {len(percorso)} punti dopo l'accorciamento")

fig = plt.figure(figsize=(18, 5.5))

# 1) In 3D: superfici degli ostacoli, A, B e il percorso
ax = fig.add_subplot(1, 3, 1, projection="3d")
v, f, _, _ = marching_cubes(np.pad(1.0 - mask, 1), level=0.5, spacing=(h, h, h))   # bordo degli ostacoli
pv = (v - h - 1.0)[:, ::-1]                                         # vertici in (x, y, z)
ax.plot_trisurf(pv[:, 0], pv[:, 1], f, pv[:, 2], color="gray", alpha=0.25, linewidth=0)
ax.plot(*percorso.T, color="red", lw=2.5, label="percorso")
ax.scatter(*A, color="green", s=60, label="A (partenza)")
ax.scatter(*B, color="blue", s=60, label="B (arrivo, sorgente)")
ax.set_box_aspect((1, 1, 1)); ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_zlim(-1, 1)
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z"); ax.legend(loc="upper left")
ax.set_title("Ostacoli (grigio) e percorso da A a B")

# 2) Fetta orizzontale all'altezza in cui il percorso attraversa il muro: u, curve di livello, ostacoli, percorso visto dall'alto
k0 = nodo(percorso[np.argmin(np.abs(percorso[:, 0]))])[0]       # strato del punto del percorso piu' vicino al muro (x = 0)
ax = fig.add_subplot(1, 3, 2)
fetta = np.ma.masked_where(mask[k0] == 0, u[k0])
ax.imshow(fetta, origin="lower", extent=(-1, 1, -1, 1), cmap="viridis")
ax.contour(x, x, fetta, levels=np.arange(0.2, 3.0, 0.2), colors="white", linewidths=0.7)
ax.imshow(np.ma.masked_where(mask[k0] == 1, mask[k0]), origin="lower", extent=(-1, 1, -1, 1), cmap="gray", vmin=-1, vmax=1)
ax.plot(percorso[:, 0], percorso[:, 1], "r-", lw=1.5)
ax.plot(*A[:2], "go"); ax.plot(*B[:2], "bo")
ax.set_title(f"Fetta z = {x[k0]:.2f}: u (colori), ostacoli (grigio)\npercorso visto dall'alto (rosso)")
ax.set_xlabel("x"); ax.set_ylabel("y")

# 3) u lungo il percorso: deve scendere con pendenza -1 (|grad u| = 1 quando F = 1)
fitto = np.concatenate([p + np.linspace(0, 1, 20, endpoint=False)[:, None] * (q - p)
                        for p, q in zip(percorso[:-1], percorso[1:])] + [percorso[-1:]])   # punti fitti sui segmenti
s = np.concatenate([[0], np.cumsum(np.linalg.norm(np.diff(fitto, axis=0), axis=1))])
uu = np.array([u_in(q) for q in fitto])
ax = fig.add_subplot(1, 3, 3)
ax.plot(s, uu, "r-", label="u lungo il percorso")
ax.plot(s, u[iA] - s, "k--", lw=1, label="pendenza -1 (ideale)")
ax.set_xlabel("strada percorsa da A"); ax.set_ylabel("u = distanza rimanente fino a B")
ax.set_title("u lungo il percorso"); ax.grid(alpha=0.3); ax.legend()

plt.tight_layout()
plt.savefig("fim3d_percorso.png", dpi=120)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import Image, display
from skimage.measure import marching_cubes

# Da eseguire DOPO la cella del percorso (usa u, mask, libero, x, h, A, B, percorso, k0)
alto = u[libero].max() * 1.05
w = np.where(libero, u, alto)                                       # negli ostacoli un valore alto ma finito (per marching_cubes)
t_max = u[libero].max()                                             # istante in cui l'onda ha coperto tutto
n_onda, n_drone = 20, 20                                            # fotogrammi: espansione dell'onda, poi volo del drone

# Superficie degli ostacoli (fissa) e percorso campionato a passo costante (per il volo del drone)
vo, fo, _, _ = marching_cubes(np.pad(1.0 - mask, 1), level=0.5, spacing=(h, h, h), step_size=2)   # step_size=2: meno triangoli, piu' veloce
po = (vo - h - 1.0)[:, ::-1]
lung = np.concatenate([[0], np.cumsum(np.linalg.norm(np.diff(percorso, axis=0), axis=1))])
s_drone = np.linspace(0, lung[-1], n_drone)
volo = np.array([np.interp(s_drone, lung, percorso[:, c]) for c in range(3)]).T   # posizioni del drone

fig = plt.figure(figsize=(11, 5), dpi=80)
ax3 = fig.add_subplot(1, 2, 1, projection="3d")
ax2 = fig.add_subplot(1, 2, 2)

def disegna(frame):
    ax3.cla(); ax2.cla()
    if frame < n_onda:                                              # fase 1: l'onda parte da B
        t = t_max * (frame + 1) / n_onda
        titolo = f"Onda partita da B: fronte u = {t:.2f}"
        fine = 0
    else:                                                           # fase 2: il drone vola da A a B
        t = t_max
        fine = frame - n_onda + 1
        titolo = f"Il drone segue il percorso ({fine}/{n_drone})"
    # 3D: ostacoli, fronte u = t, A, B, percorso
    ax3.plot_trisurf(po[:, 0], po[:, 1], fo, po[:, 2], color="gray", alpha=0.15, linewidth=0)
    if t < t_max * 0.999:
        v, f, _, _ = marching_cubes(w, level=t, spacing=(h, h, h), step_size=2)
        pv = (v - 1.0)[:, ::-1]
        ax3.plot_trisurf(pv[:, 0], pv[:, 1], f, pv[:, 2], color="tab:cyan", alpha=0.5, linewidth=0)
    ax3.scatter(*A, color="green", s=50); ax3.scatter(*B, color="blue", s=50)
    if fine > 0:
        ax3.plot(*volo[:fine].T, color="red", lw=2.5)
        ax3.scatter(*volo[fine - 1], color="red", s=80)
    ax3.set_xlim(-1, 1); ax3.set_ylim(-1, 1); ax3.set_zlim(-1, 1); ax3.set_box_aspect((1, 1, 1))
    ax3.set_xlabel("x"); ax3.set_ylabel("y"); ax3.set_zlabel("z"); ax3.set_title(titolo)
    # Fetta: zona gia' raggiunta dall'onda (colori), fronte (bianco), ostacoli (grigio), percorso visto dall'alto
    raggiunto = np.ma.masked_where((mask[k0] == 0) | (u[k0] > t), u[k0])
    ax2.imshow(raggiunto, origin="lower", extent=(-1, 1, -1, 1), cmap="viridis", vmin=0, vmax=t_max)
    ax2.imshow(np.ma.masked_where(mask[k0] == 1, mask[k0]), origin="lower", extent=(-1, 1, -1, 1), cmap="gray", vmin=-1, vmax=1)
    if t < t_max * 0.999:
        ax2.contour(x, x, np.where(mask[k0] == 1, u[k0], np.nan), levels=[t], colors="white", linewidths=1.5)
    ax2.plot(*A[:2], "go"); ax2.plot(*B[:2], "bo")
    if fine > 0:
        ax2.plot(volo[:fine, 0], volo[:fine, 1], "r-", lw=2); ax2.plot(*volo[fine - 1, :2], "ro", ms=8)
    ax2.set_xlim(-1, 1); ax2.set_ylim(-1, 1); ax2.set_xlabel("x"); ax2.set_ylabel("y")
    ax2.set_title(f"Fetta z = {x[k0]:.2f} (vista dall'alto)")

anim = animation.FuncAnimation(fig, disegna, frames=n_onda + n_drone)
anim.save("fim3d_animazione.gif", writer=animation.PillowWriter(fps=5))   # un solo calcolo dei fotogrammi
plt.close(fig)
display(Image(filename="fim3d_animazione.gif"))                     # la GIF si vede nel notebook e si puo' scaricare